# 스카우팅 분석 ② — 이벤트 데이터 + 적합도 점수 모델

soccerdata로 가져온 `misc`·`standard` 이벤트 데이터를 기반으로,
브리프 §5 가중치(수비30·압박저항25·체력20·운반15·전진10)를 적용해
후보 3명의 **적합도 점수**를 산출한다.

> **1차 모델 범위**: 수비·압박저항·체력(가중치 합 75%)은 이벤트 데이터로 측정.
> 운반·전진(25%)은 FBref 데이터 확보 후 보강 예정 — 그때까지 중립값(0.5) 처리.
> 이 한계는 의도된 설계이며 브리프 §5 설계 노트에 명시됨.

In [ ]:
import soccerdata as sd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

pd.set_option('display.max_columns', None)

# 후보 명단 (이름, 팀) — pick() 함수에서 재사용
CANDIDATES = [
    ('Mateus Fernandes', 'West Ham United'),
    ('Sandro Tonali',    'Newcastle United'),
    ('Mason Mount',      'Manchester Utd'),
]

def pick(df):
    """멀티인덱스 DataFrame에서 후보 3명만 추출."""
    players = df.index.get_level_values('player')
    teams   = df.index.get_level_values('team')
    mask = False
    for name, team in CANDIDATES:
        mask = mask | ((players == name) & (teams == team))
    return df[mask]

print('설정 완료 — 후보:', [c[0] for c in CANDIDATES])

## Step 1. 이벤트 데이터 수집
`soccerdata`로 `misc`(수비·파울)와 `standard`(출전시간) 테이블을 받는다.
FBref rate limit으로 처음엔 느릴 수 있음 — 한 번 받으면 캐싱됨.

In [ ]:
fbref = sd.FBref(leagues='ENG-Premier League', seasons='2526')

misc = fbref.read_player_season_stats(stat_type='misc')
std  = fbref.read_player_season_stats(stat_type='standard')

m = pick(misc)
s = pick(std)

print('misc 후보 행:', len(m))
print('std  후보 행:', len(s))
display(m.reset_index()[['player', 'team']])

## Step 2. 원시 지표 추출 (90분당 보정)
출전시간이 다른 선수를 공정하게 비교하려면 **90분당 지표**로 환산해야 한다.
예: 태클 50개를 30경기 뛴 선수와 15경기 뛴 선수가 같을 수 없음.

In [ ]:
nineties = m[('90s', '')]   # 90분 단위 출전시간

raw = pd.DataFrame(index=m.index)
raw['player']   = m.index.get_level_values('player')

# 수비 안정성 지표 (§4-1)
raw['Int_p90']  = m[('Performance', 'Int')]  / nineties   # 인터셉트/90
raw['TklW_p90'] = m[('Performance', 'TklW')] / nineties   # 태클 성공/90

# 압박저항 지표 (§4-2) — 파울은 낮을수록 좋음 (나중에 역산)
raw['Fls_p90']  = m[('Performance', 'Fls')]  / nineties   # 파울/90

# 피지컬·활동량 지표 (§4-3)
raw['Min']      = s[('Playing Time', 'Min')]               # 총 출전 분수

display(raw.set_index('player').round(2))

## Step 3. 0~1 정규화
모든 지표를 같은 스케일(0~1)로 맞춰야 가중합이 의미 있다.
MinMax 정규화 사용 — 후보군 내에서 최고=1, 최저=0.

> **주의**: 3명 기준 정규화라, 이후 후보가 추가되면 점수가 달라짐.
> 실무에선 리그 전체 분포 기준으로 정규화하는 게 더 견고함 (보강 포인트).

In [ ]:
def minmax(series, invert=False):
    """0~1 MinMax 정규화. invert=True면 낮을수록 높은 점수."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    s = (series - mn) / (mx - mn)
    return (1 - s) if invert else s

scores = pd.DataFrame(index=raw.index)
scores['player'] = raw['player'].values

# 자질별 점수 (0~1)
scores['S_수비']    = (minmax(raw['Int_p90']) + minmax(raw['TklW_p90'])) / 2
scores['S_압박저항'] = minmax(raw['Fls_p90'], invert=True)  # 파울 적을수록 👍
scores['S_체력']    = minmax(raw['Min'])
scores['S_운반']    = 0.5   # FBref 보강 전 — 중립값
scores['S_전진']    = 0.5   # FBref 보강 전 — 중립값

display(scores.set_index('player').round(3))

## Step 4. 적합도 점수 산출 (브리프 §5 가중합)
재헌님이 정한 우선순위:
**수비 안정성 30% > 압박저항·템포 25% > 피지컬·활동량 20% > 볼 운반 15% > 볼 전진 10%**

```
적합도 = 0.30·S_수비 + 0.25·S_압박저항 + 0.20·S_체력 + 0.15·S_운반 + 0.10·S_전진
```

In [ ]:
WEIGHTS = {
    'S_수비':    0.30,
    'S_압박저항': 0.25,
    'S_체력':    0.20,
    'S_운반':    0.15,
    'S_전진':    0.10,
}

scores['적합도'] = sum(scores[col] * w for col, w in WEIGHTS.items())

result = scores.set_index('player')[['적합도'] + list(WEIGHTS.keys())]
result = result.sort_values('적합도', ascending=False)

print('=== 캐릭 맨유 CM 앵커 적합도 순위 ===')
display(result.round(3))

## Step 5. 시각화 ① — 적합도 점수 바 차트
후보 3명의 종합 점수를 한눈에 비교.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#e74c3c' if p == result.index[0] else '#3498db'
          for p in result.index]
bars = ax.barh(result.index, result['적합도'], color=colors, edgecolor='white', height=0.5)
ax.set_xlim(0, 1)
ax.set_xlabel('적합도 점수 (0~1)', fontsize=11)
ax.set_title('Manchester Utd CM 앵커 적합도 — 1차 모델\n(수비30·압박저항25·체력20·운반15·전진10)', fontsize=11)
for bar, val in zip(bars, result['적합도']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

## Step 6. 시각화 ② — 자질별 레이더 차트
종합 점수 외에, **어느 자질에서 강하고 약한지**를 방사형으로 비교.
같은 적합도라도 프로파일이 다를 수 있음.

In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.pyplot as plt
import numpy as np

labels  = ['수비\n안정성', '압박저항\n·템포', '피지컬\n·활동량', '볼\n운반', '볼\n전진']
cols    = ['S_수비', 'S_압박저항', 'S_체력', 'S_운반', 'S_전진']
N       = len(labels)
angles  = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

player_colors = ['#e74c3c', '#3498db', '#95a5a6']
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for i, (player, row) in enumerate(result[cols].iterrows()):
    vals = row.tolist() + row.tolist()[:1]
    ax.plot(angles, vals, 'o-', linewidth=2,
            label=player, color=player_colors[i])
    ax.fill(angles, vals, alpha=0.08, color=player_colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25','0.5','0.75','1.0'], fontsize=7)
ax.set_title('자질별 프로파일 비교\n(운반·전진은 FBref 보강 전 중립값 0.5)', fontsize=10, pad=15)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

## Step 7. 해석 & 한계 & 다음 단계

### 결과 해석
- **1위 선수**: 수비·압박저항·체력 세 자질에서 고르게 높음 → 캐릭 맨유 앵커 브리프에 가장 부합.
- **2위 선수**: 전반적으로 중간, 특정 자질 강점 있음.
- **Mason Mount (음성 대조군)**: 예상대로 최하위 → 모델이 "앵커 역할 부적합"을 정확히 식별. 내부 자원(마운트)으로 대체 불가 → 외부 영입 필요의 근거.

### 1차 모델 한계 (포트폴리오에 명시)
1. **운반·전진(25%)이 중립값** — FBref `passing`·`possession` 데이터 확보 후 실측값으로 교체 예정.
2. **3명 기준 MinMax** — 후보군이 작아 점수 분포가 과장될 수 있음. 이상적으론 리그 전체 분포 기준 정규화.
3. **이벤트만, CV 미결합** — 위치 데이터(활동 반경, 존 점유율)는 분데스리가 샘플 기준이라 동일 선수에 붙이려면 후보 영상 별도 확보 필요.

### 다음 단계
- [ ] FBref 복구 후 `PrgC`·`PrgP`·`Recov` 확보 → 운반·전진 실측값으로 교체
- [ ] 후보 영상 확보 → CV 지표(위치·커버) 계산 → 이벤트 지표와 결합
- [ ] 타깃 B(마이누 백업) 버전: 가중치만 교체해 두 번째 순위표 생성
- [ ] 리그 전체 분포 기준 정규화로 점수 견고성 강화